# **Archivo 3 — Clasificación SANOS vs ENFERMOS con una CNN 3D sobre la densidad de tractos témporo-límbicos**

**Entrada:** por cada paciente, el tensor de 8 canales (`tensor_8canales.npz`: arcuato, uncinado, MdLF e ILF, izquierdo y derecho) generado en la Sección 8 del Archivo 2, más el `.tt.gz` de ese paciente, del que solo se lee la transformación `trans_to_mni`.
**Cohorte:** SANOS S1–S19 sin S9 (18) y ENFERMOS E1–E71.
**Salida:** métricas de clasificación por validación cruzada **por paciente**, con un modelo de referencia (regresión logística sobre descriptores simples) para comparar.

> **Advertencia de alcance.** Con 89 pacientes y desbalance 18 : 71, la CNN es un ejercicio exploratorio. Las métricas se reportan con intervalos de confianza y junto a un modelo de referencia; un buen AUC aquí no demuestra por sí solo un biomarcador. Además, si los SANOS y los ENFERMOS se adquirieron con protocolos o equipos distintos, el modelo podría aprender esa diferencia y no la enfermedad (Sección 2.4).

## **0. Configuración**

* `BASE_DIR`: carpeta con una subcarpeta por paciente (`S1`, `E1`, …). La etiqueta se toma de la primera letra del nombre (`S` = sano, `E` = enfermo). Cada subcarpeta debe contener `tensor_8canales.npz` (puede estar dentro de `imagenes_cnn/3d/`) y un archivo `*.tt.gz`.
* `EXCLUIR`: pacientes que no se usan (S9).
* `RES_MM`: tamaño del vóxel de la rejilla común. Con 2 mm la CNN entra en la memoria de una GPU de Colab.
* `FORMA_DESTINO`: tamaño (X, Y, Z) de la rejilla común en vóxeles. Su centro se calcula con la cohorte (Sección 2.2).

In [ ]:
import os, re, glob, json, time, warnings
import numpy as np, pandas as pd
import scipy.io as sio, gzip
from scipy import ndimage
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in __import__("sys").modules or os.path.isdir("/content")
if IN_COLAB and not os.path.isdir("/content/drive/MyDrive"):
    try:
        from google.colab import drive; drive.mount("/content/drive")
    except Exception as e:
        print("No se montó Drive:", e)

BASE_DIR = "/content/drive/MyDrive/MAESTRIA/Semestre III/Trabajo Integrador/cohorte"   # <- ajuste esta ruta
EXCLUIR = {"S9"}
RES_MM = 2.0
FORMA_DESTINO = (96, 112, 96)      # vóxeles de RES_MM en (X, Y, Z) del espacio destino
SEMILLA = 42
K_FOLDS = 5
EPOCAS_MAX = 60
CARPETA_SALIDA3 = ("/content/" if IN_COLAB else "./") + "salida_cnn/"
os.makedirs(CARPETA_SALIDA3, exist_ok=True)
rng_global = np.random.default_rng(SEMILLA)
print("Base:", BASE_DIR, "| existe:", os.path.isdir(BASE_DIR))

## **1. Inventario de la cohorte**

Se busca, por paciente, el tensor de 8 canales y el `.tt.gz`. Los pacientes sin alguno de los dos se listan y se excluyen. Se verifica el tamaño esperado de la cohorte (18 sanos y 71 enfermos) y se muestra el desbalance.

In [ ]:
def inventariar(base):
    filas = []
    for carpeta in sorted(glob.glob(os.path.join(base, "*"))):
        if not os.path.isdir(carpeta): continue
        pid = os.path.basename(carpeta)
        m = re.match(r"^([SsEe])\d+", pid)
        if not m or pid.upper() in EXCLUIR: continue
        tens = sorted(glob.glob(os.path.join(carpeta, "**", "tensor_8canales.npz"), recursive=True))
        tts  = sorted(glob.glob(os.path.join(carpeta, "**", "*.tt.gz"), recursive=True))
        filas.append(dict(paciente=pid.upper(), etiqueta=int(m.group(1).upper() == "E"),
                          grupo="ENFERMO" if m.group(1).upper() == "E" else "SANO",
                          ruta_tensor=tens[0] if tens else None, ruta_tt=tts[0] if tts else None))
    return pd.DataFrame(filas)

df_coh = inventariar(BASE_DIR)
if len(df_coh):
    faltan = df_coh[df_coh.ruta_tensor.isna() | df_coh.ruta_tt.isna()]
    if len(faltan): print("Pacientes incompletos (se excluyen):\n", faltan[["paciente", "ruta_tensor", "ruta_tt"]].to_string())
    df_coh = df_coh.dropna(subset=["ruta_tensor", "ruta_tt"]).reset_index(drop=True)
    print(df_coh.grupo.value_counts().to_string())
    esperado = {"SANO": 18, "ENFERMO": 71}
    for g, n in esperado.items():
        if (df_coh.grupo == g).sum() != n: print(f"AVISO: se esperaban {n} {g}S y hay {(df_coh.grupo == g).sum()}.")
else:
    print("No se encontraron pacientes en BASE_DIR; revise la ruta.")

## **2. Preprocesamiento: espacio común**

### **2.1. Por qué hace falta**
Los tensores del Archivo 2 están en el espacio **nativo** de cada paciente: la cabeza está en posiciones y orientaciones distintas. Una CNN compara posiciones entre pacientes, así que sin llevarlos a un espacio común aprendería diferencias de colocación de la cabeza y no de los tractos.

### **2.2. Cómo se hace**
El `.tt.gz` guarda `trans_to_mni`, una matriz afín 4×4 que lleva las coordenadas de vóxel nativas al espacio de la plantilla de DSI Studio. Esa matriz es solo la parte **lineal** del registro, por lo que la alineación es aproximada (no corrige diferencias de forma locales).

1. Se lee `trans_to_mni` (matriz $A$) de cada paciente.
2. Se define una rejilla destino de `FORMA_DESTINO` vóxeles de `RES_MM` mm, centrada en la mediana, entre pacientes, del centro del volumen transformado.
3. Cada canal se remuestrea con interpolación lineal: el vóxel destino $p$ toma el valor del vóxel nativo $A^{-1}p$.

### **2.3. Control de calidad del registro**
Se calcula, por paciente, el **centro de masa de los tractos** (densidad sumada de los 8 canales) y se transforma con $A$. Si el registro es coherente, esos centros caen cerca unos de otros en el espacio destino. Se marcan los pacientes a más de 15 mm de la mediana: suelen indicar un `trans_to_mni` mal calculado, un tractograma incompleto o una anatomía muy distinta. Esta comprobación no sustituye a abrir algunos casos en un visor.

In [ ]:
def leer_afin(ruta_tt):
    # Solo se leen las dos variables necesarias; no se decodifican las streamlines.
    m = sio.loadmat(gzip.open(ruta_tt), variable_names=["trans_to_mni", "dimension"])
    A = np.asarray(m["trans_to_mni"], dtype=np.float64).ravel()
    if A.size != 16: raise ValueError(f"trans_to_mni inesperada en {ruta_tt}: {A.size} valores")
    return A.reshape(4, 4), tuple(int(v) for v in np.asarray(m["dimension"]).ravel())

def cargar_tensor(ruta):
    z = np.load(ruta, allow_pickle=True)
    canales = [str(c) for c in z["canales"]]
    x = z["x"]
    orden = np.argsort(canales)                      # mismo orden de canales en todos los pacientes
    return x[orden], [canales[i] for i in orden]

def centroide_tractos(ruta_tensor):
    # Centro de masa (en vóxeles nativos) de la densidad sumada de los 8 canales.
    x, _ = cargar_tensor(ruta_tensor)
    tot = x.sum(axis=0)
    return np.array(ndimage.center_of_mass(tot)) if tot.sum() > 0 else np.array([np.nan] * 3)

def centro_mapeado(A, centroide_vox):
    return (A @ np.r_[centroide_vox, 1])[:3]

def calcular_centro_destino(mapeados):
    return np.nanmedian(np.array(mapeados), axis=0)

def remuestrear_a_destino(vol, A, centro_dest, forma=FORMA_DESTINO, res=RES_MM):
    # Coordenadas destino de cada vóxel de la rejilla, en las unidades de A (1 mm por vóxel nativo).
    ejes = [(np.arange(n) - (n - 1) / 2) * res + c for n, c in zip(forma, centro_dest)]
    P = np.stack(np.meshgrid(*ejes, indexing="ij"), axis=0).reshape(3, -1)
    Ainv = np.linalg.inv(A)
    src = Ainv[:3, :3] @ P + Ainv[:3, 3:4]
    out = ndimage.map_coordinates(vol, src, order=1, mode="constant", cval=0.0)
    return out.reshape(forma).astype(np.float32)

In [ ]:
if len(df_coh):
    afines, dims = [], []
    for r in df_coh.itertuples():
        A, d = leer_afin(r.ruta_tt); afines.append((A, d))
    centroides = [centroide_tractos(r.ruta_tensor) for r in df_coh.itertuples()]
    cm = np.array([centro_mapeado(A, c) for (A, d), c in zip(afines, centroides)])
    centro_dest = calcular_centro_destino(cm)
    desv = np.linalg.norm(cm - centro_dest, axis=1)
    df_coh["desv_centro_mm"] = desv
    print("Centro de los tractos en el espacio destino (mediana):", np.round(centro_dest, 1))
    print("Desviación del centro de los tractos por paciente: mediana %.1f mm, máxima %.1f mm" % (np.median(desv), desv.max()))
    marcados = df_coh[df_coh.desv_centro_mm > 15]
    print("Marcados (>15 mm):", list(marcados.paciente) or "ninguno")
    # la rejilla debe contener el cerebro: se comprueba que el tensor remuestreado conserve la masa de streamlines
    perdida = []
    for r, (A, d) in list(zip(df_coh.itertuples(), afines))[:5]:
        x, _ = cargar_tensor(r.ruta_tensor)
        y = remuestrear_a_destino(x[0], A, centro_dest)
        perdida.append(1 - y.sum() * (RES_MM ** 3) / max(x[0].sum(), 1e-9))
    print("Pérdida de señal por recorte/interpolación en 5 pacientes (fracción):", np.round(perdida, 3))

### **2.4. Construcción del conjunto de datos**
Cada paciente produce un arreglo `(8, X, Y, Z)` en la rejilla común. Se guarda en `float16` para ahorrar memoria y se cachea en disco. Se calculan también los **descriptores simples** (suma de densidad por canal y número de vóxeles ocupados) que usa el modelo de referencia.

**Sesgo por adquisición:** si los grupos se escanearon con protocolos distintos, la suma de densidad por canal puede reflejar el protocolo. Se muestra esa suma por grupo para detectar diferencias grandes antes de entrenar.

In [ ]:
RUTA_CACHE = CARPETA_SALIDA3 + "cohorte_mni.npz"
if len(df_coh):
    if os.path.exists(RUTA_CACHE):
        z = np.load(RUTA_CACHE, allow_pickle=True)
        X, y, ids, canales = z["X"], z["y"], list(z["ids"]), list(z["canales"])
        print("Cargado desde caché:", X.shape)
    else:
        X, y, ids = [], [], []
        for r, (A, d) in zip(df_coh.itertuples(), afines):
            x, canales = cargar_tensor(r.ruta_tensor)
            X.append(np.stack([remuestrear_a_destino(c, A, centro_dest) for c in x]).astype(np.float16))
            y.append(r.etiqueta); ids.append(r.paciente)
        X = np.stack(X); y = np.array(y)
        np.savez_compressed(RUTA_CACHE, X=X, y=y, ids=np.array(ids), canales=np.array(canales))
        print("Conjunto construido:", X.shape, "| canales:", canales)
    desc = np.concatenate([X.astype(np.float32).sum(axis=(2, 3, 4)), (X > 0).sum(axis=(2, 3, 4)).astype(np.float32)], axis=1)
    nombres_desc = [f"suma_{c}" for c in canales] + [f"vox_{c}" for c in canales]
    df_desc = pd.DataFrame(desc, columns=nombres_desc); df_desc["grupo"] = np.where(y == 1, "ENFERMO", "SANO")
    print(df_desc.groupby("grupo")[nombres_desc[:len(canales)]].median().round(1).T.to_string())

## **3. Validación**

Con 89 pacientes no se separa un único conjunto de prueba (tendría muy pocos sanos). Se usa **validación cruzada estratificada por paciente** repetida: en cada pliegue, ningún paciente aparece a la vez en entrenamiento y prueba, y cada pliegue conserva la proporción de sanos y enfermos. Dentro de cada pliegue de entrenamiento se aparta un 15 % de pacientes para detener el entrenamiento (*early stopping*); el pliegue de prueba no interviene en ninguna decisión del entrenamiento.

Como cada paciente es una sola muestra, no hay fuga entre cortes o *streamlines* de una misma persona.

In [ ]:
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def pliegues(y, k=K_FOLDS, semilla=SEMILLA):
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=semilla)
    return list(skf.split(np.zeros(len(y)), y))

def metricas(y_true, p, umbral=0.5):
    pred = (p >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return dict(AUC=roc_auc_score(y_true, p) if len(set(y_true)) == 2 else np.nan,
                BalancedAcc=balanced_accuracy_score(y_true, pred),
                Sensibilidad=tp / max(tp + fn, 1), Especificidad=tn / max(tn + fp, 1))

def bootstrap_ic(y_true, p, B=1000, semilla=SEMILLA):
    rng = np.random.default_rng(semilla); n = len(y_true); aucs = []
    for _ in range(B):
        i = rng.integers(0, n, n)
        if len(set(y_true[i])) == 2: aucs.append(roc_auc_score(y_true[i], p[i]))
    return np.percentile(aucs, [2.5, 97.5])

### **3.1. Modelo de referencia (regresión logística)**
Usa solo los descriptores simples de la Sección 2.4 (16 números por paciente). **La CNN solo es útil si supera claramente este modelo.** Se calcula con los mismos pliegues que la CNN.

In [ ]:
if len(df_coh):
    folds = pliegues(y)
    oof_base = np.zeros(len(y))
    for tr, te in folds:
        mod = make_pipeline(StandardScaler(), LogisticRegression(C=0.3, class_weight="balanced", max_iter=2000))
        mod.fit(desc[tr], y[tr]); oof_base[te] = mod.predict_proba(desc[te])[:, 1]
    res_base = metricas(y, oof_base); ic = bootstrap_ic(y, oof_base)
    print("Referencia (regresión logística), predicciones fuera de pliegue:")
    print({k: round(v, 3) for k, v in res_base.items()}, "| IC95%% AUC: [%.3f, %.3f]" % tuple(ic))

## **4. CNN 3D**

**Arquitectura** (pequeña a propósito, por el tamaño de la muestra): 4 bloques `Conv3d(3×3×3) → GroupNorm → ReLU → MaxPool`, con 16, 32, 64 y 64 filtros, *pooling* global promedio, *dropout* 0.5 y una neurona de salida. Unos 180 mil parámetros.

**Entrenamiento:** pérdida BCE con peso de clase (compensa el desbalance 18 : 71), AdamW con *weight decay*, *early stopping* sobre el AUC de validación interna.

**Aumento de datos:** traslaciones aleatorias de ±4 vóxeles y variación de intensidad ±10 %. **No se voltea izquierda-derecha**, porque la lateralidad puede ser parte de la señal en epilepsia.

In [ ]:
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader
    TORCH_OK = True
except ImportError:
    TORCH_OK = False; print("PyTorch no está instalado (en Colab ya viene incluido).")

if TORCH_OK:
    DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
    print("Dispositivo:", DISPOSITIVO)

    class CNN3D(nn.Module):
        def __init__(self, n_in=8, filtros=(16, 32, 64, 64), p_drop=0.5):
            super().__init__()
            capas, c = [], n_in
            for f in filtros:
                capas += [nn.Conv3d(c, f, 3, padding=1), nn.GroupNorm(4, f), nn.ReLU(inplace=True), nn.MaxPool3d(2)]
                c = f
            self.features = nn.Sequential(*capas)
            self.drop = nn.Dropout(p_drop); self.fc = nn.Linear(c, 1)
        def forward(self, x):
            h = self.features(x).mean(dim=(2, 3, 4))
            return self.fc(self.drop(h)).squeeze(1)

    class DatosTractos(Dataset):
        def __init__(self, X, y, aumentar=False):
            self.X, self.y, self.aumentar = X, y, aumentar
        def __len__(self): return len(self.y)
        def __getitem__(self, i):
            x = torch.from_numpy(self.X[i].astype(np.float32))
            if self.aumentar:
                s = np.random.randint(-4, 5, size=3)
                x = torch.roll(x, shifts=tuple(int(v) for v in s), dims=(1, 2, 3))
                x = x * float(np.random.uniform(0.9, 1.1))
            return x, torch.tensor(float(self.y[i]))

    def predecir(modelo, X, lote=8):
        modelo.eval(); out = []
        with torch.no_grad():
            for i in range(0, len(X), lote):
                xb = torch.from_numpy(X[i:i + lote].astype(np.float32)).to(DISPOSITIVO)
                out.append(torch.sigmoid(modelo(xb)).cpu().numpy())
        return np.concatenate(out)

    def entrenar_pliegue(Xtr, ytr, semilla, epocas=EPOCAS_MAX, paciencia=12, lote=4):
        torch.manual_seed(semilla); np.random.seed(semilla)
        sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=semilla)
        i_tr, i_va = next(sss.split(np.zeros(len(ytr)), ytr))
        modelo = CNN3D(n_in=Xtr.shape[1]).to(DISPOSITIVO)
        pos_w = torch.tensor([(ytr[i_tr] == 0).sum() / max((ytr[i_tr] == 1).sum(), 1)], dtype=torch.float32).to(DISPOSITIVO)
        crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
        opt = torch.optim.AdamW(modelo.parameters(), lr=1e-3, weight_decay=1e-2)
        dl = DataLoader(DatosTractos(Xtr[i_tr], ytr[i_tr], aumentar=True), batch_size=lote, shuffle=True)
        mejor, mejor_estado, sin_mejora = -1, None, 0
        for ep in range(epocas):
            modelo.train()
            for xb, yb in dl:
                xb, yb = xb.to(DISPOSITIVO), yb.to(DISPOSITIVO)
                opt.zero_grad(); crit(modelo(xb), yb).backward(); opt.step()
            pv = predecir(modelo, Xtr[i_va])
            auc = roc_auc_score(ytr[i_va], pv) if len(set(ytr[i_va])) == 2 else 0.5
            if auc > mejor:
                mejor, sin_mejora = auc, 0
                mejor_estado = {k: v.detach().clone() for k, v in modelo.state_dict().items()}
            else:
                sin_mejora += 1
                if sin_mejora >= paciencia: break
        modelo.load_state_dict(mejor_estado)
        return modelo, mejor

### **4.1. Validación cruzada de la CNN**
Los resultados se guardan por pliegue y se reúnen las predicciones **fuera de pliegue** (cada paciente se predice una sola vez con un modelo que no lo vio). Si la ejecución se interrumpe, se puede relanzar: los pliegues ya calculados se leen de disco.

In [ ]:
if TORCH_OK and len(df_coh):
    oof_cnn = np.full(len(y), np.nan); filas = []
    for k, (tr, te) in enumerate(folds):
        ruta_k = CARPETA_SALIDA3 + f"pliegue_{k}.npz"
        if os.path.exists(ruta_k):
            z = np.load(ruta_k); oof_cnn[te] = z["p"]; print(f"Pliegue {k}: cargado de disco"); continue
        t0 = time.time()
        modelo, auc_val = entrenar_pliegue(X[tr], y[tr], semilla=SEMILLA + k)
        p = predecir(modelo, X[te]); oof_cnn[te] = p
        # importancia por canal: caída del AUC al barajar ese canal entre los pacientes de prueba
        rng_imp = np.random.default_rng(SEMILLA + k); auc0 = roc_auc_score(y[te], p); caida = []
        for c in range(X.shape[1]):
            Xp = X[te].copy(); Xp[:, c] = Xp[rng_imp.permutation(len(te)), c]
            caida.append(auc0 - roc_auc_score(y[te], predecir(modelo, Xp)))
        np.savez(ruta_k, p=p, te=te, caida=np.array(caida))
        m = metricas(y[te], p); m.update(pliegue=k, AUC_val_interna=auc_val, segundos=time.time() - t0); filas.append(m)
        print(f"Pliegue {k}: AUC prueba {m['AUC']:.3f} | AUC val. interna {auc_val:.3f} | {m['segundos']:.0f} s")
    res_cnn = metricas(y, oof_cnn); ic_cnn = bootstrap_ic(y, oof_cnn)
    print("\nCNN, predicciones fuera de pliegue:", {k: round(v, 3) for k, v in res_cnn.items()}, "| IC95%% AUC: [%.3f, %.3f]" % tuple(ic_cnn))
    np.save(CARPETA_SALIDA3 + "oof_cnn.npy", oof_cnn)

## **5. Resultados y comparación**

La tabla y la curva ROC comparan la CNN con el modelo de referencia sobre los **mismos pacientes y pliegues**. El intervalo de confianza del AUC es un *bootstrap* sobre pacientes (1000 remuestreos). Si los intervalos se solapan ampliamente, no hay evidencia de que la CNN sea mejor que el modelo simple.

In [ ]:
from sklearn.metrics import roc_curve
if len(df_coh):
    filas = [dict(modelo="Regresión logística (referencia)", **res_base, IC95_AUC="[%.3f, %.3f]" % tuple(ic))]
    curvas = {"Referencia": oof_base}
    if TORCH_OK and not np.isnan(oof_cnn).any():
        filas.append(dict(modelo="CNN 3D", **res_cnn, IC95_AUC="[%.3f, %.3f]" % tuple(ic_cnn))); curvas["CNN 3D"] = oof_cnn
    df_res = pd.DataFrame(filas).round(3); print(df_res.to_string(index=False))
    df_res.to_csv(CARPETA_SALIDA3 + "resultados_modelos.csv", index=False, encoding="utf-8-sig")
    fig, ax = plt.subplots(figsize=(5, 5))
    for nombre, p in curvas.items():
        fpr, tpr, _ = roc_curve(y, p); ax.plot(fpr, tpr, lw=2, label=f"{nombre} (AUC {roc_auc_score(y, p):.2f})")
    ax.plot([0, 1], [0, 1], "k--", lw=1); ax.set_xlabel("1 − especificidad"); ax.set_ylabel("Sensibilidad")
    ax.set_title("ROC fuera de pliegue (validación cruzada por paciente)"); ax.legend(loc="lower right")
    plt.tight_layout(); plt.savefig(CARPETA_SALIDA3 + "roc_modelos.png", dpi=200); plt.show()

### **5.1. Prueba de permutación (¿el AUC supera al azar?)**
Se repite la validación del modelo de referencia con las etiquetas barajadas (200 veces). La fracción de barajados con AUC igual o mayor al observado da un valor p empírico. Es una prueba rápida del modelo simple; para la CNN completa se recomienda repetirla con pocas permutaciones (10–20) en GPU.

In [ ]:
if len(df_coh):
    def auc_cv_base(yy, semilla):
        o = np.zeros(len(yy))
        for tr, te in pliegues(yy, semilla=semilla):
            m = make_pipeline(StandardScaler(), LogisticRegression(C=0.3, class_weight="balanced", max_iter=2000)).fit(desc[tr], yy[tr])
            o[te] = m.predict_proba(desc[te])[:, 1]
        return roc_auc_score(yy, o)
    obs = roc_auc_score(y, oof_base); rng = np.random.default_rng(SEMILLA)
    nulo = np.array([auc_cv_base(rng.permutation(y), SEMILLA + i) for i in range(200)])
    pval = (1 + (nulo >= obs).sum()) / (1 + len(nulo))
    print(f"AUC observado (referencia): {obs:.3f} | AUC nulo medio {nulo.mean():.3f} | p de permutación = {pval:.3f}")

## **6. Importancia de canales**
Dentro de cada pliegue (Sección 4.1) se mide cuánto cae el AUC de la CNN al **barajar entre pacientes un canal** (tracto y lado) del conjunto de prueba. Un canal cuya permutación baja mucho el AUC es importante para el modelo. Es una medida del modelo, no una prueba de que ese tracto esté alterado biológicamente. Con pocos pacientes de prueba por pliegue, la medida es ruidosa: se muestra la media y el rango entre pliegues.

In [ ]:
if TORCH_OK and len(df_coh) and not np.isnan(oof_cnn).any():
    caidas = np.array([np.load(CARPETA_SALIDA3 + f"pliegue_{k}.npz")["caida"] for k in range(len(folds))])
    df_imp = pd.DataFrame({"canal": canales, "caida_AUC_media": caidas.mean(0), "min": caidas.min(0), "max": caidas.max(0)}).sort_values("caida_AUC_media", ascending=False)
    print(df_imp.round(3).to_string(index=False)); df_imp.to_csv(CARPETA_SALIDA3 + "importancia_canales.csv", index=False)
    fig, ax = plt.subplots(figsize=(6, 3.5)); ax.barh(df_imp.canal[::-1], df_imp.caida_AUC_media[::-1], color="#4c78a8")
    ax.set_xlabel("Caída media del AUC al barajar el canal"); plt.tight_layout(); plt.savefig(CARPETA_SALIDA3 + "importancia_canales.png", dpi=200); plt.show()

## **7. Limitaciones**

1. **Muestra pequeña y desbalanceada** (18 sanos, 71 enfermos): los intervalos son amplios y la especificidad es la métrica más inestable.
2. **Registro lineal:** `trans_to_mni` no corrige diferencias de forma; parte de la variabilidad entre pacientes es de alineación y no biológica.
3. **Sesgo de adquisición:** sin información de escáner/protocolo por grupo no se puede descartar que el modelo distinga el protocolo.
4. **Heterogeneidad clínica:** los ENFERMOS agrupan distintos tipos de epilepsia y lateralidad; una sola etiqueta binaria puede esconder subgrupos.
5. **Validación interna:** la validación cruzada no sustituye a una cohorte externa.
6. **Tractos derivados de Autotrack con n = 1 validado:** la validación de las reglas de nombrado se hizo solo en S1 (Archivo 2).